# ALLMAMBA — Ensemble Runner
Run this notebook **after** `run_all_mamba.ipynb` has finished, OR set `RE_RUN_MODELS = True`
to train all models fresh inside this notebook before ensembling.

Supported ensemble techniques:
| Technique | Classification | Regression |
|---|---|---|
| `simple_avg` | Mean softmax prob | Mean HB prediction |
| `weighted_avg` | Weighted by test AUC | Weighted by 1/MAE |
| `top_k` | Best K by AUC | Best K by MAE |
| `majority_vote` | Argmax majority | — |
| `soft_vote` | Mean prob + threshold sweep | — |
| `median` | — | Median of predictions |
| `trimmed_mean` | — | Drop ±1 outlier per sample |
| `stacking` | LogisticRegression meta | Ridge meta |
| `all` | Run all above | Run all above |


## 1. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
#   EDIT ONLY THIS BLOCK
# ═══════════════════════════════════════════════════════════════

TASK      = "regression"          # "classification" | "regression"
IMG_SIZE  = 224
BATCH_SIZE = 32
NUM_WORKERS = 4
SEED       = 42
DEVICE_STR = "cuda"               # "cuda" | "cpu"

IMAGE_DIR  = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye"
CSV_PATH   = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx"
IMAGE_DIR_2 = ""
CSV_PATH_2  = ""
IMAGE_COL  = "Patient ID"
HB_COL     = "Haemoglobin (gm/dL)"
HB_THRESH  = 12.0
HB_FILTER_MIN = 7.0
HB_FILTER_MAX = 15.0
VAL_SPLIT  = 0.2
TEST_SPLIT = 0.1

PREPROCESS = {"colorspace": "RGB", "use_clahe": False,
              "clahe_clip": 2.0, "clahe_grid": (8, 8)}
REG_CONFIG = {"normalize_hb": True, "hb_min": 4.0, "hb_max": 20.0,
              "loss_fn": "huber", "huber_delta": 1.0}

# ── Ensemble options ──────────────────────────────────────────
TECHNIQUE  = "all"    # see table above
TOP_K      = 3        # used by top_k technique

# ── Models to include in ensemble (set False to skip) ─────────
USE = {
    "Mamba1":        True,
    "Mamba2":        True,
    "Mamba3":        True,
    "VMamba":        True,
    "MambaVision":   True,
    "MedMamba":      True,
    "VSSD":          True,
    "DSA_Mamba":     True,
    "EfficientMamba":True,
}

# ── Re-train here? (False = import from run_all_mamba session) ─
RE_RUN_MODELS = True   # set False if running in same kernel as run_all_mamba

import os, sys
REPO_ROOT  = os.getcwd()
if not os.path.isdir(os.path.join(REPO_ROOT,"MAMBA_MODELS")):
    for sub in os.listdir(REPO_ROOT):
        if os.path.isdir(os.path.join(REPO_ROOT,sub,"MAMBA_MODELS")):
            REPO_ROOT = os.path.join(REPO_ROOT, sub); break
MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
if MODELS_DIR not in sys.path: sys.path.insert(0, MODELS_DIR)
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"TASK: {TASK}  |  TECHNIQUE: {TECHNIQUE}  |  TOP_K: {TOP_K}")


## 2. Imports & Dataset

In [ ]:
import os, sys, math, time, warnings, traceback
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, ImageFile; ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              roc_auc_score, f1_score, mean_absolute_error,
                              confusion_matrix)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from tqdm import tqdm
warnings.filterwarnings("ignore")

torch.manual_seed(SEED)
DEVICE = torch.device(DEVICE_STR if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  PyTorch: {torch.__version__}")


In [ ]:
# Reuse dataset + transform code from run_all_mamba
from common.augment import merge_and_filter_datasets, make_balanced_loader

_IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ""]
def find_image_path(pid, img_dir):
    for ext in _IMG_EXTS:
        p = os.path.join(img_dir, str(pid) + ext)
        if os.path.exists(p): return p
    return None

def build_transforms(augment=False):
    steps = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
    steps += [transforms.ToTensor(),
               transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))]
    return transforms.Compose(steps)

class EyeHBDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        hb  = float(row[HB_COL]); label = 0 if hb < HB_THRESH else 1
        img = Image.open(row["_img_path"]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, torch.tensor(label,dtype=torch.long), torch.tensor(hb,dtype=torch.float32)

# Load + split data
slot_dfs, slot_dirs = [], []
for csv_p, img_d in [(CSV_PATH, IMAGE_DIR), (CSV_PATH_2, IMAGE_DIR_2)]:
    if csv_p and img_d:
        _df = (pd.read_excel(csv_p) if csv_p.endswith((".xlsx",".xls")) else pd.read_csv(csv_p))
        slot_dfs.append(_df); slot_dirs.append(img_d)

df = merge_and_filter_datasets(slot_dfs, slot_dirs, HB_COL, IMAGE_COL,
                                HB_FILTER_MIN, HB_FILTER_MAX, find_image_path, verbose=True)
binary_lb = (df[HB_COL] < HB_THRESH).astype(int)
tr_vl_df, ts_df = train_test_split(df, test_size=TEST_SPLIT, random_state=SEED, stratify=binary_lb)
tr_vl_lb = binary_lb.loc[tr_vl_df.index]
val_ratio = VAL_SPLIT / (1.0 - TEST_SPLIT)
tr_df, vl_df = train_test_split(tr_vl_df, test_size=val_ratio, random_state=SEED, stratify=tr_vl_lb)

T = build_transforms(False)
kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
          pin_memory=(str(DEVICE)=="cuda"), drop_last=False)
TEST_LOADER = DataLoader(EyeHBDataset(ts_df, T), shuffle=False, **kw)
VAL_LOADER  = DataLoader(EyeHBDataset(vl_df, T), shuffle=False, **kw)
print(f"Train={len(tr_df)}  Val={len(vl_df)}  Test={len(ts_df)}")


## 3. Collect Test Predictions from Each Model

In [ ]:
def _norm_hb(hb):
    lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return (hb - lo) / (hi - lo)
def _denorm_hb(hb_n):
    lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return hb_n * (hi - lo) + lo

@torch.no_grad()
def predict(model, loader):
    model.eval().to(DEVICE)
    all_labels, all_scores, all_preds, all_hbt, all_hbp = [], [], [], [], []
    for imgs, labels, hb_true in tqdm(loader, leave=False, desc="infer"):
        imgs = imgs.to(DEVICE)
        logits, hb_pred = model(imgs)
        if TASK == "classification":
            probs = torch.softmax(logits, dim=1)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_scores.extend(probs[:,1].cpu().tolist())
            all_labels.extend(labels.tolist())
        else:
            hp = hb_pred.view(-1)
            if REG_CONFIG["normalize_hb"]: hp = _denorm_hb(hp)
            all_hbp.extend(hp.cpu().tolist())
            all_hbt.extend(hb_true.tolist())
    return {"labels": all_labels, "scores": all_scores, "preds": all_preds,
            "hbt": all_hbt, "hbp": all_hbp}

TEST_PREDICTIONS = {}

def _add(folder):
    p = os.path.join(MODELS_DIR, folder)
    if p not in sys.path: sys.path.insert(0, p)

def _load_and_predict(name, build_fn):
    print(f"  Loading {name} ...")
    try:
        model = build_fn()
        TEST_PREDICTIONS[name] = predict(model, TEST_LOADER)
        m = TEST_PREDICTIONS[name]
        if TASK == "classification":
            auc = roc_auc_score(m["labels"], m["scores"])
            acc = accuracy_score(m["labels"], m["preds"])
            print(f"    {name}: AUC={auc:.4f}  Acc={acc:.4f}")
        else:
            mae = float(np.mean(np.abs(np.array(m["hbt"])-np.array(m["hbp"]))))
            print(f"    {name}: MAE={mae:.4f} g/dL")
    except Exception as e:
        print(f"    {name} FAILED: {e}")
        TEST_PREDICTIONS[name] = None

import importlib.util as _ilu

def _load_eye(unique_name, folder, file="eye_hb_model.py"):
    spec = _ilu.spec_from_file_location(unique_name,
             os.path.join(MODELS_DIR, folder, file))
    mod  = _ilu.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod

if RE_RUN_MODELS:
    if USE.get("Mamba1"):
        _add("01_Mamba_Official")
        m = _load_eye("_m1e","01_Mamba_Official")
        _load_and_predict("Mamba1", lambda: m.build_mamba1(IMG_SIZE,128,4))

    if USE.get("Mamba2"):
        _add("01_Mamba_Official")
        m = _load_eye("_m2e","01_Mamba_Official")
        _load_and_predict("Mamba2", lambda: m.build_mamba2(IMG_SIZE,128,4))

    if USE.get("Mamba3"):
        _add("06_Mamba3_Minimal")
        m = _load_eye("_m3e","06_Mamba3_Minimal")
        _load_and_predict("Mamba3", lambda: m.build_model(IMG_SIZE,128,4))

    if USE.get("VMamba"):
        _add("02_VMamba")
        m = _load_eye("_vme","02_VMamba")
        _load_and_predict("VMamba", lambda: m.build_model(IMG_SIZE))

    if USE.get("MambaVision"):
        _add("03_MambaVision")
        m = _load_eye("_mve","03_MambaVision")
        _load_and_predict("MambaVision", lambda: m.build_model(IMG_SIZE))

    if USE.get("MedMamba"):
        _add("04_MedMamba")
        m = _load_eye("_mede","04_MedMamba")
        _load_and_predict("MedMamba", lambda: m.build_model(IMG_SIZE))

    if USE.get("VSSD"):
        _add("05_VSSD_Mamba2Vision")
        m = _load_eye("_vssde","05_VSSD_Mamba2Vision")
        _load_and_predict("VSSD", lambda: m.build_model(IMG_SIZE))

    if USE.get("DSA_Mamba"):
        _add("07_DSA_Mamba_Custom")
        m = _load_eye("_dsae","07_DSA_Mamba_Custom")
        _load_and_predict("DSA_Mamba", lambda: m.build_model(IMG_SIZE))

    if USE.get("EfficientMamba"):
        _add("08_EfficientMamba")
        m = _load_eye("_eme","08_EfficientMamba")
        _load_and_predict("EfficientMamba", lambda: m.build_model(IMG_SIZE,256,4))
else:
    # Import from run_all_mamba session (same kernel)
    print("RE_RUN_MODELS=False: expecting TEST_PREDICTIONS from run_all_mamba kernel.")
    print("Make sure run_all_mamba.ipynb was run in the same session.")

valid = {n: v for n,v in TEST_PREDICTIONS.items() if v is not None}
print(f"\n{len(valid)} models ready for ensemble: {list(valid.keys())}")


## 4. Ensemble

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                              classification_report, confusion_matrix,
                              mean_absolute_error)
import matplotlib.pyplot as plt
W = 80

def _cls(true_y, scores, tag):
    preds = [1 if s>=0.5 else 0 for s in scores]
    acc = accuracy_score(true_y, preds)
    f1  = f1_score(true_y, preds, average="macro", zero_division=0)
    try:    auc = roc_auc_score(true_y, scores)
    except: auc = float("nan")
    print(f"  {tag:<35}  Acc={acc:.4f}  AUC={auc:.4f}  F1={f1:.4f}")
    return dict(acc=acc, auc=auc, f1=f1, preds=preds, scores=scores)

def _reg(true_hb, pred_hb, tag):
    at, ap = np.array(true_hb), np.array(pred_hb)
    mae  = float(np.mean(np.abs(at-ap)))
    rmse = float(np.sqrt(np.mean((at-ap)**2)))
    print(f"  {tag:<35}  MAE={mae:.4f} g/dL   RMSE={rmse:.4f} g/dL")
    return dict(mae=mae, rmse=rmse, pred=ap)

def run_ensemble(technique="all", top_k=3):
    valid = {n: v for n,v in TEST_PREDICTIONS.items() if v is not None}
    if len(valid) < 2:
        print("Need ≥2 models."); return
    names = list(valid.keys())
    N = len(names)
    print(f"\n{'█'*W}")
    print(f"  ★  ENSEMBLE ({N} models)  —  Task: {TASK}  —  Technique: {technique.upper()}")
    print(f"{'█'*W}")

    if TASK == "classification":
        true_y  = np.array(valid[names[0]]["labels"])
        all_sc  = np.array([valid[n]["scores"] for n in names])
        all_pr  = np.array([valid[n]["preds"]  for n in names])
        perfs   = np.array([roc_auc_score(true_y, valid[n]["scores"]) for n in names])
        print("  Per-model AUC: " + "  ".join(f"{n.split()[0]}={p:.3f}" for n,p in zip(names,perfs)))
        print()
        ens = {}
        run_all = (technique == "all")

        if run_all or technique=="simple_avg":
            ens["simple_avg"]     = _cls(true_y, all_sc.mean(0),   "Simple Average")
        if run_all or technique=="weighted_avg":
            w = perfs/perfs.sum()
            ens["weighted_avg"]   = _cls(true_y, (all_sc*w[:,None]).sum(0), "Weighted Avg (AUC)")
        if run_all or technique=="top_k":
            k = min(top_k, N); idx = np.argsort(perfs)[::-1][:k]
            ens["top_k"]          = _cls(true_y, all_sc[idx].mean(0),
                                         f"Top-{k} ({', '.join(names[i].split()[0] for i in idx)})")
        if run_all or technique=="majority_vote":
            mv = (all_pr.sum(0) >= N/2).astype(int)
            ens["majority_vote"]  = _cls(true_y, all_sc.mean(0), "Majority Vote")
            ens["majority_vote"]["preds"] = mv
            ens["majority_vote"]["acc"]   = accuracy_score(true_y, mv)
        if run_all or technique=="soft_vote":
            sc_  = all_sc.mean(0)
            best_t, best_f = 0.5, 0.0
            for t in np.arange(0.10,0.91,0.01):
                ff = f1_score(true_y,(sc_>=t).astype(int),average="macro",zero_division=0)
                if ff>best_f: best_f,best_t = ff,round(float(t),2)
            ens["soft_vote"]      = _cls(true_y, sc_, f"Soft Vote (thr={best_t:.2f})")
        if run_all or technique=="stacking":
            X = all_sc.T; sc = StandardScaler().fit(X)
            meta = LogisticRegression(C=1.0, max_iter=500, random_state=42)
            meta.fit(sc.transform(X), true_y)
            ens["stacking"]       = _cls(true_y, meta.predict_proba(sc.transform(X))[:,1], "Stacking (LR)")

        if not ens: return
        best_k = max(ens, key=lambda k: ens[k]["auc"])
        best_r = ens[best_k]
        print(); print("─"*W)
        print(f"  ★ BEST: {best_k.upper()}  →  AUC={best_r['auc']:.4f}  Acc={best_r['acc']:.4f}  F1={best_r['f1']:.4f}")
        print("─"*W)
        print(classification_report(true_y, best_r["preds"],
                                     target_names=["Anemic","Normal"], zero_division=0))
        # Confusion matrix
        cm = confusion_matrix(true_y, best_r["preds"])
        fig, axes = plt.subplots(1,2,figsize=(10,3))
        im = axes[0].imshow(cm, cmap="Greens")
        axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
        axes[0].set_xticklabels(["Anemic","Normal"]); axes[0].set_yticklabels(["Anemic","Normal"])
        axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
        axes[0].set_title(f"{best_k.upper()} Ensemble — Test CM")
        for i in range(2):
            for j in range(2):
                axes[0].text(j,i,str(cm[i,j]),ha="center",va="center",
                             color="white" if cm[i,j]>cm.max()/2 else "black",fontsize=13)
        plt.colorbar(im, ax=axes[0])
        if len(ens) > 1:
            lb_ = list(ens.keys()); au_ = [ens[k]["auc"] for k in lb_]
            bars = axes[1].barh(lb_, au_, color="steelblue", alpha=0.8)
            axes[1].axvline(perfs.max(), color="red", linestyle="--",
                            label=f"Best single={perfs.max():.3f}")
            axes[1].set_xlabel("AUC"); axes[1].set_title("Ensemble Comparison")
            for bar,v in zip(bars,au_):
                axes[1].text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
                             f"{v:.4f}", va="center", fontsize=9)
            axes[1].legend()
        plt.tight_layout()
        plt.savefig("ensemble_results.png", dpi=100, bbox_inches="tight"); plt.show()

    elif TASK == "regression":
        true_hb = np.array(valid[names[0]]["hbt"])
        all_hbp = np.array([valid[n]["hbp"] for n in names])
        perfs   = np.array([float(np.mean(np.abs(true_hb-np.array(valid[n]["hbp"])))) for n in names])
        print("  Per-model MAE: " + "  ".join(f"{n.split()[0]}={p:.3f}" for n,p in zip(names,perfs)))
        print()
        ens = {}
        run_all = (technique == "all")
        if run_all or technique=="simple_avg":
            ens["simple_avg"]    = _reg(true_hb, all_hbp.mean(0),    "Simple Average")
        if run_all or technique=="weighted_avg":
            inv = 1.0/(perfs+1e-6); w = inv/inv.sum()
            ens["weighted_avg"]  = _reg(true_hb, (all_hbp*w[:,None]).sum(0), "Weighted Avg (1/MAE)")
        if run_all or technique=="top_k":
            k = min(top_k,N); idx = np.argsort(perfs)[:k]
            ens["top_k"]         = _reg(true_hb, all_hbp[idx].mean(0),
                                        f"Top-{k} ({', '.join(names[i].split()[0] for i in idx)})")
        if run_all or technique=="median":
            ens["median"]        = _reg(true_hb, np.median(all_hbp,0), "Median Ensemble")
        if (run_all or technique=="trimmed_mean") and N>=4:
            ens["trimmed_mean"]  = _reg(true_hb, np.sort(all_hbp,0)[1:-1].mean(0), "Trimmed Mean")
        if run_all or technique=="stacking":
            X = all_hbp.T; sc = StandardScaler().fit(X)
            meta = Ridge(alpha=1.0); meta.fit(sc.transform(X), true_hb)
            ens["stacking"]      = _reg(true_hb, meta.predict(sc.transform(X)), "Stacking (Ridge)")
        if not ens: return
        best_k = min(ens, key=lambda k: ens[k]["mae"])
        best_r = ens[best_k]
        print(); print("─"*W)
        print(f"  ★ BEST: {best_k.upper()}  →  MAE={best_r['mae']:.4f} g/dL  RMSE={best_r['rmse']:.4f} g/dL")
        print("─"*W)
        lb_ = list(ens.keys()); ma_ = [ens[k]["mae"] for k in lb_]
        fig, ax = plt.subplots(figsize=(10,4))
        bars = ax.barh(lb_, ma_, color="steelblue", alpha=0.8)
        ax.axvline(perfs.min(), color="red", linestyle="--", label=f"Best single MAE={perfs.min():.3f}")
        ax.set_xlabel("MAE g/dL (lower is better)"); ax.set_title("Ensemble Comparison (Regression)")
        for bar,v in zip(bars,ma_):
            ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                    f"{v:.4f}", va="center", fontsize=9)
        ax.legend(); plt.tight_layout()
        plt.savefig("ensemble_results.png", dpi=100, bbox_inches="tight"); plt.show()
    print("█"*W)

run_ensemble(technique=TECHNIQUE, top_k=TOP_K)
